In [3]:
import pandas as pd
import numpy as np

# Chargement du fichier DVF
df = pd.read_csv('../data/ValeursFoncieres-2025.txt',
                  sep='|',
                  encoding='utf-8',
                  low_memory=False)

print(f"Dimensions : {df.shape}")
print(f"\nColonnes :")
print(df.columns.tolist())
print(f"\nPremières lignes :")
df.head()

Dimensions : (3714829, 43)

Colonnes :
['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']

Premières lignes :


,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Surface Carrez du 5eme lot,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,78.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,3.0,Dépendance,NaN,0.0,0.0,S,NaN,133.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,1.0,Maison,NaN,111.0,5.0,S,NaN,133.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,S,NaN,46.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,17.0


In [4]:
# Filtrer uniquement le département 06 - Alpes-Maritimes
print("Colonnes disponibles :")
print(df.columns.tolist())

# Chercher la colonne département
dept_col = [col for col in df.columns if 'partement' in col or 'code' in col.lower()]
print(f"\nColonnes liées au département : {dept_col}")

Colonnes disponibles :
['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']

Colonnes liées au département : ['Code voie', 'Code postal', 'Code departement', 'Code commune', 'Code type local']


In [5]:
# Filtrer département 06 - Alpes-Maritimes
df_06 = df[df['Code departement'] == '06'].copy()

print(f"Transactions département 06 : {len(df_06)}")
print(f"\nTypes de biens :")
print(df_06['Type local'].value_counts())
print(f"\nCommunes principales :")
print(df_06['Commune'].value_counts().head(10))

Transactions département 06 : 86929

Types de biens :
Type local
Dépendance                                  39378
Appartement                                 24399
Maison                                       6018
Local industriel. commercial ou assimilé     3263
Name: count, dtype: int64

Communes principales :
Commune
NICE                    22562
ANTIBES                  7938
CANNES                   7815
GRASSE                   3827
LE CANNET                3540
CAGNES SUR MER           3424
MANDELIEU-LA-NAPOULE     2698
MENTON                   2694
VALLAURIS                2627
SAINT LAURENT DU VAR     2516
Name: count, dtype: int64


In [6]:
# Garder uniquement Appartements et Maisons
df_immo = df_06[df_06['Type local'].isin(['Appartement', 'Maison'])].copy()

# Nettoyer la valeur foncière
df_immo['Valeur fonciere'] = df_immo['Valeur fonciere'].astype(str).str.replace(',', '.').str.replace(' ', '')
df_immo['Valeur fonciere'] = pd.to_numeric(df_immo['Valeur fonciere'], errors='coerce')

# Garder les colonnes utiles
df_immo = df_immo[[
    'Date mutation', 'Nature mutation', 'Valeur fonciere',
    'Code postal', 'Commune', 'Type local',
    'Surface reelle bati', 'Nombre pieces principales'
]].copy()

# Supprimer les valeurs manquantes
df_immo = df_immo.dropna(subset=['Valeur fonciere', 'Surface reelle bati'])

# Filtrer les prix aberrants
df_immo = df_immo[df_immo['Valeur fonciere'] > 10000]
df_immo = df_immo[df_immo['Valeur fonciere'] < 5000000]
df_immo = df_immo[df_immo['Surface reelle bati'] > 10]
df_immo = df_immo[df_immo['Surface reelle bati'] < 500]

# Calculer le prix au m²
df_immo['Prix_m2'] = (df_immo['Valeur fonciere'] / df_immo['Surface reelle bati']).round(2)

print(f"Dataset nettoyé : {df_immo.shape}")
print(f"\nPrix moyen au m² par commune :")
print(df_immo.groupby('Commune')['Prix_m2'].mean().round(0).sort_values(ascending=False).head(10))

Dataset nettoyé : (28903, 9)

Prix moyen au m² par commune :
Commune
SAINT-JEAN-CAP-FERRAT    17736.0
EZE                      17361.0
SAINT-PAUL-DE-VENCE      16174.0
CAP-D AIL                13356.0
BEAUSOLEIL               13105.0
VILLEFRANCHE-SUR-MER     11963.0
BEAULIEU SUR MER         11448.0
CANNES                   11393.0
LA TURBIE                10767.0
CHATEAUNEUF              10258.0
Name: Prix_m2, dtype: float64


In [7]:
# Prix moyen à Nice par arrondissement
print("Prix moyen au m² à Nice :")
nice = df_immo[df_immo['Commune'] == 'NICE']
print(f"Prix moyen Nice : {nice['Prix_m2'].mean().round(0)} €/m²")
print(f"Prix médian Nice : {nice['Prix_m2'].median().round(0)} €/m²")
print(f"Transactions Nice : {len(nice)}")

print(f"\nPrix moyen par type de bien :")
print(df_immo.groupby('Type local')['Prix_m2'].mean().round(0))

print(f"\nPrix moyen par type de bien à Nice :")
print(nice.groupby('Type local')['Prix_m2'].mean().round(0))

# Sauvegarde
df_immo.to_csv('../data/immobilier_clean.csv', index=False)
print("\nDataset sauvegardé !")

Prix moyen au m² à Nice :
Prix moyen Nice : 8854.0 €/m²
Prix médian Nice : 5111.0 €/m²
Transactions Nice : 8645

Prix moyen par type de bien :
Type local
Appartement    7875.0
Maison         7743.0
Name: Prix_m2, dtype: float64

Prix moyen par type de bien à Nice :
Type local
Appartement    8909.0
Maison         7731.0
Name: Prix_m2, dtype: float64

Dataset sauvegardé !
